# Retrain Roof Symbol Detector (Google Colab)

Retrains the YOLOv8 roof symbol detector on the cluster-augmented dataset.

**Before running:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Have `roof_package.zip` ready on your computer — it contains everything:
   training data, validation data, testing data, and the model to fine-tune from.

Run the cells top to bottom. The upload cell shows a **Choose Files** button.
At the end you download the retrained `best.pt`.

In [ ]:
# 1. Install and check GPU
!pip -q install ultralytics

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU - go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# 2. Upload roof_package.zip (training + testing data + model, all in one)
import zipfile
from pathlib import Path

from google.colab import files

if not Path("roof_package").is_dir():
    uploaded = files.upload()  # choose roof_package.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")

MODEL_PATH = Path("roof_package/symbol_detector_best.pt")
DATA_YAML = "roof_package/data/data.yaml"

# Rewrite data.yaml with ABSOLUTE paths: Ultralytics resolves relative paths
# against its own working/datasets dir, not the yaml location, which breaks
# on Colab with "images not found, missing path '/content/valid/images'".
data_dir = Path("roof_package/data").resolve()
Path(DATA_YAML).write_text(
    f"train: {(data_dir / 'train' / 'images').as_posix()}\n"
    f"val: {(data_dir / 'valid' / 'images').as_posix()}\n"
    f"test: {(data_dir / 'test' / 'images').as_posix()}\n"
    "nc: 9\n"
    "names: ['CURB', 'EXHAUST FAN', 'HATCH DETAIL', 'HEAT DETAIL', 'PIPE HOUSING', "
    "'PIPE PENETRATION', 'PLUMBING STACK', 'ROOF DRAIN', 'SKYLIGHT']\n"
)

print("training images:  ", len(list(Path("roof_package/data/train/images").glob("*"))))
print("validation images:", len(list(Path("roof_package/data/valid/images").glob("*"))))
print("testing images:   ", len(list(Path("roof_package/data/test/images").glob("*"))))
print("model:", MODEL_PATH, "-", round(MODEL_PATH.stat().st_size / 1e6, 1), "MB")

In [ ]:
# 3. Train (fine-tune from the packaged model)
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
results = model.train(
    data=DATA_YAML,
    epochs=120,
    patience=30,       # stop early if validation stops improving
    imgsz=640,
    batch=16,
    device=0,          # GPU
    seed=20260923,
    name="roof_clusters",
    exist_ok=True,
)

BEST = "runs/detect/roof_clusters/weights/best.pt"
print("Best weights saved to:", BEST)

In [ ]:
# 4. Evaluate NEW model vs OLD model on the held-out TEST set
from ultralytics import YOLO

print("=== OLD model (before retraining) ===")
old_metrics = YOLO(str(MODEL_PATH)).val(data=DATA_YAML, split="test", device=0)

print("\n=== NEW model (after retraining) ===")
new_metrics = YOLO(BEST).val(data=DATA_YAML, split="test", device=0)

print(f"\nOLD: mAP50 {old_metrics.box.map50:.4f}  precision {old_metrics.box.mp:.4f}  recall {old_metrics.box.mr:.4f}")
print(f"NEW: mAP50 {new_metrics.box.map50:.4f}  precision {new_metrics.box.mp:.4f}  recall {new_metrics.box.mr:.4f}")

In [ ]:
# 5. Download the retrained model
# Copy the downloaded best.pt into this project's drawing-zoom-split/model/
# folder on your machine, then tell the assistant the path to deploy it.
from google.colab import files

files.download(BEST)